# Player words

Regenerates `src/data/player_words.json`, the data behind the "player words" scrolly:
for each player, the verbs where she is the grammatical subject ("she does") vs object
("done to her") and the adjectives predicated of her, scored against her peers inside Fox —
and Fox against the other outlets — with weighted log-odds (Monroe et al.'s Fightin' Words).

The Fox side reads `data/fox_wnba.csv`;
the other-outlets side pools the full-text bodies in `yahoo_sports.csv`, `espn.csv`, and
`just_womens_sports.csv`. Pronouns resolve to the nearest preceding named player; sentences
naming two players only count tokens attached directly to a name.

Needs spaCy (`pip install spacy && python -m spacy download en_core_web_sm`) and scikit-learn.
The spaCy parses are cached in `notebooks/.cache/` — the first run takes ~20 minutes,
re-runs are seconds. Delete the cache to re-parse.

In [ ]:
import csv, html, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import fighting_words as fw

DATA = Path("../data")
OUT = Path("../src/data/player_words.json")
CACHE = Path(".cache"); CACHE.mkdir(exist_ok=True)

MIN_MENTIONS = 60   # attachments a player needs to get her own column
TOP_N = 30
MIN_COUNT = 5
MAX_EXAMPLES = 3

# surname -> canonical; "thomas" (Lia) and "white" (the adjective) stay unmatchable
PLAYERS = {
    "caitlin clark": "Caitlin Clark", "clark": "Caitlin Clark",
    "angel reese": "Angel Reese", "reese": "Angel Reese",
    "brittney griner": "Brittney Griner", "griner": "Brittney Griner",
    "sophie cunningham": "Sophie Cunningham", "cunningham": "Sophie Cunningham",
    "paige bueckers": "Paige Bueckers", "bueckers": "Paige Bueckers",
    "a'ja wilson": "A'ja Wilson", "aja wilson": "A'ja Wilson",
    "napheesa collier": "Napheesa Collier", "collier": "Napheesa Collier",
    "sabrina ionescu": "Sabrina Ionescu", "ionescu": "Sabrina Ionescu",
    "dijonai carrington": "DiJonai Carrington", "carrington": "DiJonai Carrington",
    "aliyah boston": "Aliyah Boston", "boston": "Aliyah Boston",
    "alyssa thomas": "Alyssa Thomas", "diana taurasi": "Diana Taurasi",
    "taurasi": "Diana Taurasi", "kelsey plum": "Kelsey Plum", "plum": "Kelsey Plum",
    "natasha cloud": "Natasha Cloud", "breanna stewart": "Breanna Stewart",
    "kamilla cardoso": "Kamilla Cardoso", "chennedy carter": "Chennedy Carter",
}
NAME_RE = re.compile(r"\b(" + "|".join(sorted(PLAYERS, key=len, reverse=True)) + r")\b", re.I)
PRONOUN = {"she", "her", "hers"}

SKIP_VERBS = {"be", "have", "do", "say", "get", "go", "make", "take", "come", "see",
              "know", "think", "want", "look", "give", "use", "find", "tell", "become"}
SKIP_ADJ = {"other", "more", "most", "such", "same", "own", "many", "much", "few",
            "last", "next", "first", "second", "new", "old", "good", "big", "sure",
            "able", "likely", "several", "former", "recent"}

# Fox staples promo lines mid-paragraph; the all-caps run defeats spaCy's splitter
BOILERPLATE = re.compile(r"CLICK HERE[^a-z]*(?=[A-Z][a-z]|$)")

## Corpora

Fox bodies as-is; other-outlet bodies capped at 400 words apiece so Yahoo's volume
doesn't swamp the parse.

In [ ]:
csv.field_size_limit(10 ** 9)

def fox_articles():
    for row in csv.DictReader((DATA / "fox_wnba.csv").open()):
        body = BOILERPLATE.sub(" ", (row.get("body") or "")).strip()
        if len(body.split()) > 80:
            yield {"url": row.get("url", ""), "body": body}

def other_articles(limit_words=400):
    for name in ("yahoo_sports", "espn", "just_womens_sports"):
        for row in csv.DictReader((DATA / f"{name}.csv").open()):
            outlet = row.get("outlet") or ""
            body = (row.get("body") or "").strip()
            if outlet and not outlet.startswith("Fox") and len(body.split()) > 80:
                yield {"url": row.get("url", ""),
                       "body": " ".join(BOILERPLATE.sub(" ", body).split()[:limit_words])}

fox_rows = list(fox_articles())
other_rows = list(other_articles())
len(fox_rows), len(other_rows)

## Extraction

For every player token (or a pronoun resolved to her), three relations off the dependency
parse: verbs she is subject of, verbs she is object of, adjectives modifying or predicated
of her. Up to three example sentences per (player, relation, lemma), windowed around the
word itself.

In [ ]:
def adjective_form(token):
    # rejoin hyphenated compounds: "all-American" not "american"
    parts, cursor = [token.lemma_.lower()], token
    while (cursor.i >= 2 and cursor.nbor(-1).text == "-"
           and not cursor.nbor(-1).whitespace_ and not cursor.nbor(-2).whitespace_):
        cursor = cursor.nbor(-2)
        parts.insert(0, cursor.text.lower())
    return "-".join(parts)


def window_example(sent, token, budget=460):
    # centre the stored example on the word it is an example OF
    raw = sent.text
    pos = token.idx - sent.start_char
    if len(raw) <= budget:
        piece, pre, post = raw, False, False
    else:
        start = max(0, pos - budget // 2)
        end = min(len(raw), start + budget)
        start = max(0, min(start, end - budget))
        piece, pre, post = raw[start:end], start > 0, end < len(raw)
        if pre:
            piece = piece.split(" ", 1)[-1]
        if post:
            piece = piece.rsplit(" ", 1)[0]
    text = " ".join(html.unescape(piece).split())
    return ("…" if pre else "") + text + ("…" if post else "")


def extract(nlp, rows):
    found = defaultdict(lambda: defaultdict(Counter))
    examples = defaultdict(list)
    seen, texts = set(), []
    for row in rows:
        if row["url"] in seen:
            continue
        seen.add(row["url"])
        texts.append(row["body"])
    print(f"{len(texts)} articles", flush=True)

    for n, doc in enumerate(nlp.pipe(texts, batch_size=20), 1):
        if n % 2000 == 0:
            print(f"  parsed {n}/{len(texts)}", flush=True)
        recent = None   # last player named, for pronoun resolution
        for sent in doc.sents:
            names = [PLAYERS[m.group(1).lower()] for m in NAME_RE.finditer(sent.text)]
            if len(set(names)) > 1:
                # two players in one sentence: attribution is guesswork, so only
                # tokens directly attached to a name count
                recent = None
            elif names:
                recent = names[0]

            for token in sent:
                who = None
                low = token.text.lower()
                if low in PRONOUN:
                    who = recent if len(set(names)) <= 1 else None
                elif NAME_RE.fullmatch(token.text):
                    who = PLAYERS[low]
                if not who:
                    continue

                def note(relation, lemma, word_token):
                    found[who][relation][lemma] += 1
                    key = f"{who}|{relation}|{lemma}"
                    if len(examples[key]) < MAX_EXAMPLES:
                        line = window_example(sent, word_token)
                        if all(line != e["t"] for e in examples[key]):
                            examples[key].append({"t": line, "w": word_token.text.lower()})

                head = token.head
                if head.pos_ in {"VERB", "AUX"} and head.lemma_.lower() not in SKIP_VERBS:
                    if token.dep_ in {"nsubj", "csubj", "agent"}:
                        note("subject", head.lemma_.lower(), head)
                    elif token.dep_ in {"dobj", "obj", "nsubjpass", "pobj"}:
                        note("object", head.lemma_.lower(), head)

                # JJ/JJR/JJS are real adjectives; VBN is a verb wearing an adjective's hat
                for child in token.children:
                    if child.tag_.startswith("JJ") and child.lemma_.lower() not in SKIP_ADJ:
                        note("describe", adjective_form(child), child)
                if token.dep_ in {"nsubj", "nsubjpass"} and head.pos_ == "AUX":
                    for child in head.children:
                        if child.tag_.startswith("JJ") and child.dep_ in {"acomp", "attr"}:
                            if child.lemma_.lower() not in SKIP_ADJ:
                                note("describe", adjective_form(child), child)
    return found, dict(examples)

In [ ]:
def cached_parse(name, rows):
    cache = CACHE / f"{name}.json"
    if cache.exists():
        raw = json.loads(cache.read_text())
        print(f"{name}: reusing {cache} (delete it to re-parse)")
        return ({p: {r: Counter(w) for r, w in b.items()} for p, b in raw["counts"].items()},
                raw["examples"])
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
    found, examples = extract(nlp, rows)
    cache.write_text(json.dumps({
        "counts": {p: {r: dict(w) for r, w in b.items()} for p, b in found.items()},
        "examples": examples}))
    return found, examples

found, examples = cached_parse("fox_parsed", fox_rows)
other_found, other_examples = cached_parse("others_parsed", other_rows)

## Participles

spaCy's small model tags passive participles as adjectives, so "she was violently fouled"
files "fouled" under what she *is*. If the stem is a verb this corpus uses elsewhere, the
-ed form is a report of an event: it moves from "describes her" to "done to her".

In [ ]:
TRUE_PARTICIPIAL_ADJ = {
    "excited", "worried", "concerned", "disappointed", "frustrated", "injured",
    "retired", "undefeated", "unfazed", "beloved", "talented", "gifted",
    "experienced", "respected", "celebrated", "grateful", "tired", "surprised",
    "determined", "committed", "qualified", "unranked", "ranked", "seeded",
}

def stems_of(word):
    # fouled -> foul, shoved -> shove, slammed -> slam
    base = word.rsplit("-", 1)[-1]
    if not base.endswith("ed"):
        return []
    out = {base[:-2], base[:-1]}
    if len(base) > 4 and base[-3] == base[-4]:
        out.add(base[:-3])
    return [w for w in out if len(w) > 2]


def reclassify_participles(found, examples):
    verbs = set()
    for buckets in found.values():
        for relation in ("subject", "object"):
            verbs.update(buckets.get(relation, Counter()))

    moved = Counter()
    for player, buckets in found.items():
        describe = buckets.get("describe")
        if not describe:
            continue
        for word in list(describe):
            if word in TRUE_PARTICIPIAL_ADJ or not word.endswith("ed"):
                continue
            if re.search(r"-(hand|foot|side|sight|mind|will|heart)ed$", word):
                continue   # "right-handed" describes a person
            if not any(stem in verbs for stem in stems_of(word)):
                continue
            count = describe.pop(word)
            buckets.setdefault("object", Counter())[word] += count
            moved[word] += count
            old, new = f"{player}|describe|{word}", f"{player}|object|{word}"
            if old in examples:
                examples.setdefault(new, [])
                for line in examples.pop(old):
                    if line not in examples[new]:
                        examples[new].append(line)
    if moved:
        print("moved to 'done to her': "
              + ", ".join(f"{w} ({n})" for w, n in moved.most_common(8)))
    return found, examples

found, examples = reclassify_participles(found, examples)
other_found, other_examples = reclassify_participles(other_found, other_examples)

## Weighted log-odds

Fightin' Words over pseudo-documents (one per word type, repeated once per occurrence);
the vectorizer's document-frequency thresholds are opened up since every word lives in
one or two pseudo-documents. A side only gets words that lean its way AND that it
actually uses.

In [ ]:
def contrast(subject, others):
    from sklearn.feature_extraction.text import CountVectorizer

    left = [" ".join([word] * n) for word, n in subject.items()]
    right = [" ".join([word] * n) for word, n in others.items()]
    if not left or not right:
        return {"clark": [], "others": []}
    vectorizer = CountVectorizer(decode_error="ignore", min_df=1, max_df=1.0)
    scores = fw.bayes_compare_language(left, right, ngram=1, prior=0.05, cv=vectorizer)
    total = subject + others
    kept = sorted([(w, z) for w, z in scores if total[w] >= MIN_COUNT], key=lambda p: p[1])
    shape = lambda p: {"word": p[0], "z": round(p[1], 2),
                       "clark": subject.get(p[0], 0), "others": others.get(p[0], 0)}
    mine = [shape(p) for p in kept[::-1] if p[1] > 0 and subject.get(p[0], 0) > 0]
    theirs = [shape(p) for p in kept if p[1] < 0 and others.get(p[0], 0) > 0]
    return {"clark": mine[:TOP_N], "others": theirs[:TOP_N]}


def find_examples(store, who, relation, word):
    parts = ("subject", "object", "describe") if relation == "all" else (relation,)
    out = []
    for part in parts:
        out.extend(store.get(f"{who}|{part}|{word}", []))
    return out[:MAX_EXAMPLES]

## Build the payload

Per player: each relation scored against her peers pooled ("all" merges the three
relations for players whose coverage is a single episode), plus Fox-on-her against the
other outlets on her. One named pair, Clark vs Reese.

In [ ]:
RELATIONS = ("subject", "object", "describe", "all")

def bucket(store, name, relation):
    if relation != "all":
        return store.get(name, {}).get(relation, Counter())
    merged = Counter()
    for part in ("subject", "object", "describe"):
        merged.update(store.get(name, {}).get(part, Counter()))
    return merged

mentions = {name: sum(sum(c.values()) for c in b.values()) for name, b in found.items()}
cast = [n for n, total in sorted(mentions.items(), key=lambda kv: -kv[1])
        if total >= MIN_MENTIONS]
print(f"{len(cast)} players clear {MIN_MENTIONS} attachments ({len(found) - len(cast)} below)")

players = {}
for name in cast:
    entry = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        mine = bucket(found, name, relation)
        others = Counter()
        for peer in found:
            if peer != name:
                others.update(bucket(found, peer, relation))
        scored = contrast(mine, others)
        for side, owner in (("clark", name), ("others", None)):
            for word in scored[side]:
                if owner:
                    word["examples"] = find_examples(examples, owner, relation, word["word"])
                else:
                    # pooled column: examples from whichever peer uses the word most
                    best, count = None, 0
                    for peer, buckets in found.items():
                        if peer == name:
                            continue
                        got = buckets.get(relation, Counter()).get(word["word"], 0)
                        if got > count:
                            best, count = peer, got
                    word["examples"] = examples.get(
                        f"{best}|{relation}|{word['word']}", []) if best else []
                    word["who"] = best
        entry["relations"][relation] = scored
        entry["counts"][relation] = {"clark": sum(mine.values()),
                                     "others": sum(others.values())}

        theirs = bucket(other_found, name, relation)
        if sum(mine.values()) >= 25 and sum(theirs.values()) >= 25:
            versus = contrast(mine, theirs)
            for word in versus["clark"]:
                word["examples"] = find_examples(examples, name, relation, word["word"])
            for word in versus["others"]:
                word["examples"] = find_examples(other_examples, name, relation, word["word"])
            entry.setdefault("vsOutlets", {})[relation] = versus
            entry.setdefault("vsCounts", {})[relation] = {
                "clark": sum(mine.values()), "others": sum(theirs.values())}
    players[name] = entry

In [ ]:
PAIRS = [("Caitlin Clark", "Angel Reese")]
pairs = {}
for left, right in PAIRS:
    if left not in found or right not in found:
        continue
    block = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        a = bucket(found, left, relation)
        b = bucket(found, right, relation)
        if sum(a.values()) < 25 or sum(b.values()) < 25:
            continue
        scored = contrast(a, b)
        for word in scored["clark"]:
            word["examples"] = find_examples(examples, left, relation, word["word"])
        for word in scored["others"]:
            word["examples"] = find_examples(examples, right, relation, word["word"])
        block["relations"][relation] = scored
        block["counts"][relation] = {"clark": sum(a.values()), "others": sum(b.values())}
    pairs[f"{left} vs {right}"] = {**block, "left": left, "right": right}

payload = {"order": cast, "mentions": {n: mentions[n] for n in cast},
           "pairs": pairs,
           "players": players,
           "hasOutletComparison": bool(other_found),
           "source": "article bodies",
           "nDocs": len(fox_rows)}

#### Overwrites `src/data/player_words.json`

In [ ]:
# slim copy for the site: one example survives per word (hover), the rest dropped
slim = json.loads(json.dumps(payload))
for block in list(slim["players"].values()) + list(slim["pairs"].values()):
    for group in ("relations", "vsOutlets"):
        for scored in (block.get(group) or {}).values():
            for side in ("clark", "others"):
                for word in scored[side]:
                    examples_here = word.pop("examples", None) or []
                    if examples_here:
                        first = examples_here[0]
                        if isinstance(first, dict):
                            word["eg"] = first["t"]
                            word["egw"] = first["w"]
                        else:
                            word["eg"] = first

OUT.write_text(json.dumps(slim))
print(f"wrote {OUT} ({OUT.stat().st_size / 1e6:.1f} MB)")

## Preview

In [ ]:
new = json.loads(OUT.read_text())
preview = []
for name in new["order"]:
    rels = new["players"][name]["relations"]
    preview.append({
        "player": name,
        "mentions": new["mentions"][name],
        "she does": ", ".join(w["word"] for w in rels["subject"]["clark"][:5]),
        "done to her": ", ".join(w["word"] for w in rels["object"]["clark"][:5]),
        "described as": ", ".join(w["word"] for w in rels["describe"]["clark"][:5]),
    })
pd.DataFrame(preview)